# Kiểm tra dữ liệu đầy đủ trên Colab
Notebook này chỉ mount Drive, xác minh checksum, giải nén vào `/content` và chạy audit/preflight. Không tải model, không huấn luyện. Dùng runtime mới để tránh nhầm dữ liệu từ lượt chạy cũ.

In [ ]:
from pathlib import Path

from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE")
assert ROOT.is_dir(), f"Sai tài khoản Drive hoặc thiếu thư mục dự án: {ROOT}"

In [ ]:
import hashlib
import subprocess
import sys
import zipfile

SOURCE_DIR = ROOT / "source/releases"
hash_files = sorted(SOURCE_DIR.glob("source_*.sha256"))
assert hash_files, f"Không có source release: {SOURCE_DIR}"
SOURCE_HASH = hash_files[-1]
SOURCE_NAME = SOURCE_HASH.stem
SOURCE_ZIP = SOURCE_DIR / f"{SOURCE_NAME}.zip"
assert SOURCE_ZIP.is_file() and SOURCE_HASH.is_file()
expected, filename = SOURCE_HASH.read_text(encoding="utf-8").split()
assert filename == SOURCE_ZIP.name
digest = hashlib.sha256()
with SOURCE_ZIP.open("rb") as source:
    for block in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == expected, "Source ZIP checksum mismatch"
REPO = Path("/content/Vietnamese-VQA-Explanation")
assert not REPO.exists(), "Dùng runtime mới; không ghi đè source đang có"
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    names = archive.namelist()
    assert names and all(
        name.startswith("Vietnamese-VQA-Explanation/") and ".." not in Path(name).parts
        for name in names
    )
    assert archive.testzip() is None
    archive.extractall("/content")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)], check=True)
print("Source verified:", SOURCE_NAME, expected)

In [ ]:
BUNDLE = ROOT / "datasets/releases/vivqax_image_disjoint_v1"
EXPECTED_BUNDLE_SHA256 = "46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522"
WORK_DATA = Path("/content/vqa_work/vivqax_image_disjoint_v1")
assert BUNDLE.is_dir() and not WORK_DATA.exists()
assert hashlib.sha256((BUNDLE / "bundle.json").read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
subprocess.run(
    [
        sys.executable,
        str(REPO / "scripts/verify_coco_bundle.py"),
        str(BUNDLE),
        "--extract-to",
        str(WORK_DATA),
    ],
    cwd=REPO,
    check=True,
)
assert (WORK_DATA / "metadata/all.jsonl").is_file()
print("Bundle verified and extracted:", WORK_DATA)

In [ ]:
PROFILE = REPO / "configs/environments/colab_full.example.yaml"
for stage in ("audit", "preflight"):
    command = [
        sys.executable,
        "-u",
        str(REPO / "scripts/run_pipeline.py"),
        stage,
        "--environment",
        str(PROFILE),
    ]
    if stage == "preflight":
        command.append("--all-train-records")
    subprocess.run(command, cwd=REPO, check=True)
print("Full-data audit and train-record preflight passed; no GPU training performed")

Khi hai lệnh cuối qua, xem `runs/audit_*/execution.json` và `runs/preflight_*/execution.json` mới trên Drive. Audit kiểm tra ảnh/split; preflight kiểm tra 29.459 câu train và đường dẫn ảnh. Đây chưa phải kiểm tra token/VRAM hay kết quả mô hình.